In [ ]:
# This cell sets up the core tools we need for dataset loading and leakage-safe splitting.
# - sys: lets us inspect which Python interpreter the notebook is using.
# - pandas: used for reading/writing CSV files and table-like data operations.
# - Path: cleaner and safer file path handling across operating systems.
# - GroupShuffleSplit: split utility that keeps rows with the same group (id) together.
import sys
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# Sanity check: print interpreter and pandas version.
# This helps confirm whether the notebook is running inside the intended environment.
print("Interpreter:", sys.executable)
print("pandas:", pd.__version__)

In [ ]:
# Define the raw data directory relative to this notebook location.
# notebooks/EDA.ipynb -> ../data/raw
base = Path("../data/raw")

# Load the single source-of-truth dataset.
# We split from master.csv to generate train/val/test consistently.
df = pd.read_csv(base / "master.csv")

# Quick dataset diagnostics:
# - Loaded shape: total rows and columns in master dataset.
# - Unique ids: number of distinct groups used for leakage-safe splitting.
print("Loaded shape:", df.shape)
print("Unique ids:", df["id"].nunique())

In [ ]:
# LEAKAGE-SAFE SPLIT STRATEGY
# ---------------------------
# Why group-based splitting?
# The same id can appear in multiple rows (for example, multiple questions linked to one passage).
# If we split row-wise, identical ids can leak into train and validation/test sets.
# GroupShuffleSplit prevents that by assigning entire ids to exactly one split.

# Step 1: Split full data into
# - train (80%)
# - temp  (20%)
# random_state=42 makes this reproducible.
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df["id"]))

# Use integer indices produced by the splitter to build DataFrames.
train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

# Step 2: Split temp (20%) equally into val and test.
# 50% of temp = 10% of original each.
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["id"]))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Print row counts to confirm the split result.
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("Total:", len(train_df) + len(val_df) + len(test_df))

In [ ]:
# VALIDATE THAT THERE IS NO GROUP LEAKAGE
# --------------------------------------
# Convert ids from each split to Python sets.
# Set intersection gives ids appearing in both sets.
train_ids = set(train_df["id"])
val_ids = set(val_df["id"])
test_ids = set(test_df["id"])

# For a correct leakage-safe split, all three intersections should be zero.
print("train ∩ val :", len(train_ids & val_ids))
print("train ∩ test:", len(train_ids & test_ids))
print("val ∩ test  :", len(val_ids & test_ids))

In [ ]:
# SAVE OUTPUT SPLITS TO DISK
# --------------------------
# index=False avoids adding an extra unnamed integer column in the CSV.
train_df.to_csv(base / "train.csv", index=False)
val_df.to_csv(base / "val.csv", index=False)
test_df.to_csv(base / "test.csv", index=False)

# Print absolute paths so you can confirm exactly where files were written.
print("Saved:")
print((base / "train.csv").resolve())
print((base / "val.csv").resolve())
print((base / "test.csv").resolve())

In [ ]:
# DATA QUALITY CHECKS (POST-SPLIT)
# --------------------------------
# Goal: ensure train/val/test are clean enough before feature engineering and model training.
# We will check:
# 1) Missing values in critical columns
# 2) Duplicate rows
# 3) Answer-label distribution (A/B/C/D) per split
# 4) Basic text-length statistics for article/question

splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}

critical_cols = ["id", "article", "question", "A", "B", "C", "D", "answer"]

for name, split_df in splits.items():
    print("=" * 70)
    print(f"SPLIT: {name.upper()} | shape={split_df.shape}")

    # 1) Missing values in critical columns
    print("\n[1] Missing values (critical columns):")
    missing_counts = split_df[critical_cols].isna().sum().sort_values(ascending=False)
    print(missing_counts)

    # 2) Exact duplicate rows
    print("\n[2] Duplicate row count:")
    dup_count = split_df.duplicated().sum()
    print(dup_count)

    # 3) Answer-label distribution (counts + percentages)
    print("\n[3] Answer distribution:")
    answer_counts = split_df["answer"].value_counts(dropna=False).sort_index()
    answer_pct = split_df["answer"].value_counts(normalize=True, dropna=False).sort_index() * 100
    print("Counts:")
    print(answer_counts)
    print("Percent:")
    print(answer_pct.round(2))

    # 4) Text-length diagnostics (character length)
    # We cast to string so the check is robust even if a non-string sneaks in.
    print("\n[4] Text length summary (characters):")
    article_len = split_df["article"].astype(str).str.len()
    question_len = split_df["question"].astype(str).str.len()
    print("Article length:")
    print(article_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
    print("Question length:")
    print(question_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))

    print()

In [ ]:
# DATA CLEANUP: HANDLE MISSING OPTION VALUES (A/B/C/D)
# -----------------------------------------------------
# What issue did we find?
# In the quality checks, a small number of rows had missing values in option columns:
# - train split: a few missing values in A and D
# - test split: a few missing values in A and C
# - val split: no missing option values
#
# Why is this an issue?
# Our task is multiple-choice answer prediction. Each sample is expected to contain
# four complete options (A, B, C, D). If any option is missing:
# 1) feature construction may break or become inconsistent,
# 2) model input quality drops,
# 3) evaluation fairness is affected because incomplete questions are malformed.
#
# How are we fixing it?
# Since the number of affected rows is very small relative to dataset size,
# we use a simple and robust strategy: DROP only rows where any option (A/B/C/D) is missing.
# This avoids introducing synthetic text through imputation and keeps data trustworthy.

option_cols = ["A", "B", "C", "D"]

# Record sizes before cleanup for transparent reporting.
before_sizes = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
}

# Drop rows with missing values in any required option column.
train_df = train_df.dropna(subset=option_cols).copy()
val_df = val_df.dropna(subset=option_cols).copy()
test_df = test_df.dropna(subset=option_cols).copy()

# Record sizes after cleanup to quantify exactly how many rows were removed.
after_sizes = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
}

print("Rows removed due to missing A/B/C/D options:")
for split_name in ["train", "val", "test"]:
    removed = before_sizes[split_name] - after_sizes[split_name]
    print(f"- {split_name}: {removed}")

print("\nUpdated shapes after cleanup:")
print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)

# Optional safety check: confirm zero missing values remain in option columns.
print("\nRemaining missing values in option columns:")
print("train:\n", train_df[option_cols].isna().sum())
print("val:\n", val_df[option_cols].isna().sum())
print("test:\n", test_df[option_cols].isna().sum())

In [ ]:
# Persist cleaned splits to disk (overwrite old files)
base = Path("../data/raw")

train_df.to_csv(base / "train.csv", index=False)
val_df.to_csv(base / "val.csv", index=False)
test_df.to_csv(base / "test.csv", index=False)

print("Saved cleaned files:")
print((base / "train.csv").resolve())
print((base / "val.csv").resolve())
print((base / "test.csv").resolve())

In [ ]:
#Verifying that there are no missing values in the CSVs critical columns,
#basically checking if the previous steps were successful (saving the DFs to the CSVs on disk)

train_check = pd.read_csv("../data/raw/train.csv")
val_check = pd.read_csv("../data/raw/val.csv")
test_check = pd.read_csv("../data/raw/test.csv")

print("Missing A/B/C/D after save:")
for name, d in [("train", train_check), ("val", val_check), ("test", test_check)]:
    print(name, d[["A","B","C","D"]].isna().sum().to_dict())